# Exploratory Data Analysis

Comprehensive analysis of social media engagement dataset for predicting audience reaction to posts before publication.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from config import DATASET_RAW, RESULTS_FIGURES, RESULTS_TABLES
from data_loader import load_raw_dataset, clean_dataset, save_cleaned_dataset
from utils import print_dataset_overview, standardize_column_names, check_missing_values

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## 1. Load and Clean Data

In [ ]:
# Load raw dataset
df_raw = load_raw_dataset()
print(f"Raw dataset shape: {df_raw.shape}")
print(f"\nColumn names:\n{df_raw.columns.tolist()}")

In [ ]:
# Clean dataset
df_cleaned = clean_dataset(df_raw)
print(f"Cleaned dataset shape: {df_cleaned.shape}")
print(f"\nData types:\n{df_cleaned.dtypes}")

## 2. Dataset Overview

In [ ]:
print_dataset_overview(df_cleaned)

In [ ]:
# Dataset summary statistics
overview_stats = {
    'Rows': df_cleaned.shape[0],
    'Columns': df_cleaned.shape[1],
    'Numeric Features': len(df_cleaned.select_dtypes(include=[np.number]).columns),
    'Categorical Features': len(df_cleaned.select_dtypes(include=['object', 'category']).columns),
    'Missing Values': df_cleaned.isnull().sum().sum(),
    'Duplicate Rows': df_cleaned.duplicated().sum(),
    'Memory Usage (MB)': df_cleaned.memory_usage(deep=True).sum() / 1024**2
}

overview_df = pd.DataFrame(list(overview_stats.items()), columns=['Metric', 'Value'])
print(overview_df.to_string(index=False))
overview_df.to_csv(RESULTS_TABLES / 'dataset_overview.csv', index=False)

## 3. Target Variable Analysis

In [ ]:
# Analyze numeric target variables
numeric_targets = ['likes', 'comments', 'shares', 'impressions', 'reach', 'engagement_rate']

target_stats = df_cleaned[numeric_targets].describe().T
target_stats['skewness'] = df_cleaned[numeric_targets].skew()
target_stats['kurtosis'] = df_cleaned[numeric_targets].kurtosis()

print("Target Variables Statistics:")
print(target_stats)
target_stats.to_csv(RESULTS_TABLES / 'target_statistics.csv')

In [ ]:
# Distribution plots for target variables
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, col in enumerate(numeric_targets):
    axes[idx].hist(df_cleaned[col], bins=50, edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'Distribution of {col}', fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(True, alpha=0.3)
    
    # Add statistics
    mean = df_cleaned[col].mean()
    median = df_cleaned[col].median()
    axes[idx].axvline(mean, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean:.2f}')
    axes[idx].axvline(median, color='green', linestyle='--', linewidth=2, label=f'Median: {median:.2f}')
    axes[idx].legend()

plt.tight_layout()
plt.savefig(RESULTS_FIGURES / 'target_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Platform Analysis

In [ ]:
# Analyze performance by platform
platform_analysis = df_cleaned.groupby('platform')[numeric_targets].agg(['mean', 'median', 'std'])
print("\nPerformance by Platform:")
print(platform_analysis)
platform_analysis.to_csv(RESULTS_TABLES / 'platform_analysis.csv')

In [ ]:
# Platform comparison visualization
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, col in enumerate(numeric_targets):
    platform_means = df_cleaned.groupby('platform')[col].mean().sort_values(ascending=False)
    platform_means.plot(kind='bar', ax=axes[idx], color='steelblue', edgecolor='black')
    axes[idx].set_title(f'Average {col} by Platform', fontweight='bold')
    axes[idx].set_ylabel(col)
    axes[idx].set_xlabel('Platform')
    axes[idx].tick_params(axis='x', rotation=45)
    axes[idx].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(RESULTS_FIGURES / 'platform_performance.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Post Type Analysis

In [ ]:
# Analyze performance by post type
posttype_analysis = df_cleaned.groupby('post_type')[numeric_targets].mean()
print("\nPerformance by Post Type:")
print(posttype_analysis)
posttype_analysis.to_csv(RESULTS_TABLES / 'post_type_analysis.csv')

In [ ]:
# Post type comparison
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, col in enumerate(numeric_targets):
    posttype_means = df_cleaned.groupby('post_type')[col].mean().sort_values(ascending=False)
    posttype_means.plot(kind='bar', ax=axes[idx], color='coral', edgecolor='black')
    axes[idx].set_title(f'Average {col} by Post Type', fontweight='bold')
    axes[idx].set_ylabel(col)
    axes[idx].set_xlabel('Post Type')
    axes[idx].tick_params(axis='x', rotation=45)
    axes[idx].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(RESULTS_FIGURES / 'post_type_performance.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Audience Demographics Analysis

In [ ]:
# Age group analysis
if 'age_group' in df_cleaned.columns:
    age_analysis = df_cleaned.groupby('age_group')['engagement_rate'].agg(['mean', 'median', 'count'])
    print("\nEngagement by Age Group:")
    print(age_analysis)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    age_analysis['mean'].sort_values(ascending=False).plot(kind='bar', ax=ax, color='lightgreen', edgecolor='black')
    ax.set_title('Average Engagement Rate by Age Group', fontweight='bold', fontsize=14)
    ax.set_ylabel('Engagement Rate')
    ax.set_xlabel('Age Group')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(RESULTS_FIGURES / 'age_group_engagement.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Gender analysis
if 'audience_gender' in df_cleaned.columns:
    gender_analysis = df_cleaned.groupby('audience_gender')['engagement_rate'].agg(['mean', 'median', 'count'])
    print("\nEngagement by Gender:")
    print(gender_analysis)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    gender_analysis['mean'].plot(kind='bar', ax=ax, color='lightblue', edgecolor='black')
    ax.set_title('Average Engagement Rate by Gender', fontweight='bold', fontsize=14)
    ax.set_ylabel('Engagement Rate')
    ax.set_xlabel('Gender')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(RESULTS_FIGURES / 'gender_engagement.png', dpi=300, bbox_inches='tight')
    plt.show()

## 7. Correlation Analysis

In [ ]:
# Numeric correlation matrix
numeric_cols = df_cleaned.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df_cleaned[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            ax=ax, cbar_kws={'label': 'Correlation'}, square=True)
ax.set_title('Correlation Matrix of Numeric Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_FIGURES / 'correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nTop correlations with engagement_rate:")
if 'engagement_rate' in corr_matrix.columns:
    engagement_corr = corr_matrix['engagement_rate'].sort_values(ascending=False)
    print(engagement_corr)

## 8. Create Classification Targets

In [ ]:
# Create binary classification targets based on engagement rate percentiles
df_with_targets = df_cleaned.copy()

# Viral: Top 25% (75th percentile)
viral_threshold = df_cleaned['engagement_rate'].quantile(0.75)
df_with_targets['viral'] = (df_cleaned['engagement_rate'] >= viral_threshold).astype(int)

# High Engagement: Above median (50th percentile)
high_engagement_threshold = df_cleaned['engagement_rate'].quantile(0.50)
df_with_targets['high_engagement'] = (df_cleaned['engagement_rate'] >= high_engagement_threshold).astype(int)

print(f"Viral Threshold (75th percentile): {viral_threshold:.4f}")
print(f"Viral Posts: {df_with_targets['viral'].sum()} ({df_with_targets['viral'].mean()*100:.1f}%)")
print(f"\nHigh Engagement Threshold (50th percentile): {high_engagement_threshold:.4f}")
print(f"High Engagement Posts: {df_with_targets['high_engagement'].sum()} ({df_with_targets['high_engagement'].mean()*100:.1f}%)")

In [ ]:
# Sentiment distribution
if 'sentiment' in df_cleaned.columns:
    sentiment_counts = df_cleaned['sentiment'].value_counts()
    print("\nSentiment Distribution:")
    print(sentiment_counts)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    sentiment_counts.plot(kind='bar', ax=ax, color=['green', 'gray', 'red'], edgecolor='black')
    ax.set_title('Sentiment Distribution', fontweight='bold', fontsize=14)
    ax.set_ylabel('Count')
    ax.set_xlabel('Sentiment')
    ax.tick_params(axis='x', rotation=0)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(RESULTS_FIGURES / 'sentiment_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()

## 9. Missing Values and Data Quality

In [ ]:
# Check missing values
missing_info = check_missing_values(df_cleaned, return_df=True)
if len(missing_info) > 0:
    print("\nMissing Values Summary:")
    print(missing_info)
else:
    print("\nNo missing values found in cleaned dataset.")

## 10. Save Cleaned Dataset

In [ ]:
# Save cleaned dataset with targets
save_cleaned_dataset(df_with_targets)
print(f"Cleaned dataset saved with shape: {df_with_targets.shape}")
print(f"\nColumns in cleaned dataset:")
print(df_with_targets.columns.tolist())